---
title: "`pytask` Config: Defining the Pipeline Internals in `pytask`"
engine: jupyter
---

## config

> This is the config module for the `pytask` pipeline. 
This module defines the data catalog(s) and any hard-coded parameters that are used throughout the pipeline.

In [ ]:
#| default_exp config:
#

In [ ]:
#| hide:
#
from nbdev.showdoc import *

In [ ]:
#| exports: 
#

import pandas as pd

from pathlib import Path
from pyprojroot import here
from pytask import DataCatalog


SRC = here() / "src" / "era5_sandbox"
BLD = here() / "bld"

demo_catalog = DataCatalog()

## `DEV_MODE`: A Quick Development Flag

I'm adding a flag to the config that can be used for quick development. 
If you import this boolean variable, it can be used to skip tasks,
setup samples, etc. on the fly by `marking` a task with the `pytask.mark.skipif`
decorator. Change this to `False` when you're ready to run the full pipeline.

In [ ]:
#| exports:
#
DEV_MODE=True

## The Data Catalog

To manage our pipeline, we're going to use a nested data catalog structure.
This way, we can easily return specific entries to specific tasks
without having to manage multiple different data catalogs. Specifically,
we'll have a data catalog for each stage of the pipeline, and each catalog
will have entries for the inputs, outputs, and any other parameters needed
for that stage. This is similar to how we used Hydra configs, but
using the `pytask` data catalog, we can more easily gather the data
for a specific task in structured manner entirely in Python.

In [ ]:
#| exports:
#

stages = ["mydata", 'mydata2', # from the demo, ignore
    "download", # download task
    "aggregate", # aggregation task
    "publish", # publishing task
    "viz"] # visualization task

buckets = [
    "inputs", # any specific inputs, eg for carrying over between tasks
    "outputs", # specific output task returns
    "jobs", # job parameters as a dataframe
    "params" # any lingering hardcoded parameters
    ]

data_catalog = {

    stage: {bucket: DataCatalog(name=f"{stage}_{bucket}") for bucket in buckets}
    for stage in stages
}

## The Download Task

A good strategy may be to set pipeline stage parameters in the config file, 
and then use the `pytask` data catalog to manage the data. This way, we can
easily change the parameters without having to modify the code. This is especially 
useful for the API query, where we need to be able to set the parameter grid for
the years and data types we want to download data for. So, let's create an entry in the data catalog specifically for the download task.

A good strategy I thought about for grid parameter comprehension is to create a dataframe expands all the combinations of
parameters, and then uses each combination to create the tasks which are then 
easily added to the data catalog. This way, we can still easily inspect the 
pipeline and see what tasks are being run, while also being able to easily 
change the parameters in the config file without too much hassle.

An important framework decision I'm making here is that each ROW of the dataframe corresponds to a single task, so that we can quickly understand at a glance what the task is doing, and also easily develop the code for the task itself. This is different from the hydra approach where a job is first specified by a default config, and then the parameters are swept over in multiple config files. This is a more flexible approach, IMO, because:

1. each row defines a single task run, so it's easy to understand what the run is doing
2. it's easy to add or remove runs by simply expanding the list of parameters and using dataframe filters to remove irrelevant parameter combinations
3. we don't have to independently inspect and manage multiple different/overriding config files
4. it's all in Python, so we can use the full power of the language to define
   the parameters and the tasks in a single sweep, not through the need of
   hydra+snakemake multi stage/multi-lingual config system

So, to do this, we define one job as a query to the CDS API that must contain:
- The dataset (re-analysis)
- The year
- The month
- All days in the month
- All times of day (hour)
- The geography (region), which will need:
    - The URL to the shapefile to calculate the bounding box

Given one combination of all of these, a single SLURM job can complete the first "task" in parallel by having a run assigned to each row of the dataframe.

In [ ]:
#| exports:

# a dataframe for the query parameters, with nested entries for days, times, and variables
# Dimensions
years = [str(x) for x in range(2009, 2025)]  # 16 years
months = [str(x).zfill(2) for x in range(1, 13)]      # 12 months
geographies = ["madagascar", "nepal"]  # 2 geographies

# nested values; we want ALL days, times, and variables for each job
days = [str(x).zfill(2) for x in range(1, 32)]
times = [f"{x:02d}:00" for x in range(24)]
variables = ["2m_dewpoint_temperature", "2m_temperature", "total_precipitation", "volumetric_soil_water_layer_1"]

product_type = "reanalysis"

# Map shapefiles to geography
shapefiles = {
    "madagascar": "https://data.humdata.org/dataset/26fa506b-0727-4d9d-a590-d2abee21ee22/resource/ed94d52e-349e-41be-80cb-62dc0435bd34/download/mdg_adm_bngrc_ocha_20181031_shp.zip",
    "nepal": "https://data.humdata.org/dataset/07db728a-4f0f-4e98-8eb0-8fa9df61f01c/resource/2eb4c47f-fd6e-425d-b623-d35be1a7640e/download/npl_adm_nd_20240314_ab_shp.zip"
}

# Build row-wise combinations of (year, month, geography)
rows = []
for year in years:
    for month in months:
        for geo in geographies:
            rows.append({
                "year": year,
                "month": month,
                "geography": geo,
                "shapefile": shapefiles[geo],
                "product_type": product_type,
                "day": days,
                "time": times,
                "variables": variables,
                "output": f"{year}_{month}_{geo}"
            })

# Create dataframe
query_df = pd.DataFrame(rows)

In [ ]:
query_df

In [ ]:
print(f"Number of estimated jobs: {query_df.shape[0]}. Examples...")

for i, row in query_df.sample(3).iterrows():
    print(f"Year: {row['year']}, Month: {row['month']}, Geography: {row['geography']}, Link: {row['shapefile']}, Variables: {row['variables']}")

Now add them to the catalog. We're going to use a dictionary to
nest data catalogs so that we can return specific task products to
named data catalog nodes.

In [ ]:
#| export:
# set up catalog

data_catalog['download']['jobs'].add("queries_df", query_df)

Our data catalog now has a `download|jobs` node with a `queries_df` entry that contains the dataframe of all the jobs to be run in this task.

In [ ]:
data_catalog['download']['jobs']['queries_df'].load().head()

## The Aggregation Task

To carry out the aggregation, we will follow similar logic to the original pipeline and use xarray to aggregate data into spatial and temporal averages. The aggregation task will take the downloaded data and compute the mean over the specified time period and spatial region. However, in this case, we want to aggregate the data diurnally, so we will need to fetch the sundown and sunrise times for the region and use them to compute the diurnal averages.

Once again, we will use a dataframe to define the parameters for the aggregation task.

Here we will use a dataframe with the jobs as rows;
the first column is "input" which is the list of query names from
the download task, and the last column is the output object name. Columns
in between can be the parameters needed for the aggregation task, which
then get expanded to the full list of jobs with `itertools.product`, `explode` or similar,
and filtered as necessary.

For explanations of the parameters, see the Aggregation Task notebook's final `task_aggregate_data_diurnal` function.

In [ ]:
#| exports:

# aggregate task parameters

inputs = query_df["output"].tolist()
outputs = [f"{i}_agg" for i in inputs]

variable_dict = {
    "2m_dewpoint_temperature": "d2m",
    "2m_temperature": "t2m",
    "total_precipitation": "tp",
    "volumetric_soil_water_layer_1": "swvl1"
}

# list of params that get fed into the task functions
agg_params = {
    "time": ["day", "night"],
    "solar_classification": ["before"],
    "variables": variables,
    "variables_short": [variable_dict[x] for x in variables],
    "aggregation_name": ["mean", "sum", "max", "min"]
}

from itertools import product
import pandas as pd

# expand all the params
agg_params = pd.DataFrame(list(product(*agg_params.values())), columns=agg_params.keys())

Inspecting it:

In [ ]:
agg_params

Let's keep only rows where the variables and variables_short match

In [ ]:
#| exports:
# quick filter to keep only matching rows

agg_params = agg_params[agg_params.apply(lambda x: variable_dict[x['variables']] == x['variables_short'], axis=1)]

In [ ]:
agg_params

Great, and now keeping `sum` only for total precipitation (we don't need mean, max, min for that variable), and removing `sum` for all other variables (we don't need sum for temperature or soil moisture):

In [ ]:
#| exports:
# remove rows where tp aggregation is not sum
mask = (agg_params['variables_short'] == "tp") & (agg_params['aggregation_name'] != "sum")
agg_params = agg_params[~mask]

# remove rows where non-tp aggregation is sum
mask = (agg_params['variables_short'] != "tp") & (agg_params['aggregation_name'] == "sum")
agg_params = agg_params[~mask]

In [ ]:
agg_params

Now we add the input and output columns by joining:

In [ ]:
#| exports:
# set up inputs and parameters
inputs = pd.DataFrame({"input": inputs})
aggregate_jobs = inputs.merge(agg_params, how="cross")

This result gives us the full list of jobs for the aggregation task. 20 rows for the parameters,
and 384 inputs/outputs, giving a total of 7680 jobs:

In [ ]:
assert aggregate_jobs.shape[0] == 20 * len(inputs)
aggregate_jobs

A few more configuration items need to be added, like
the local timezone for each geography, the healthshed filename,
the healthshed unique ID variable name in the shapefile,
and whether the variable is instantaneous or accumulated:

In [ ]:
#| exports:
# add a few more columns
aggregate_jobs['local_tz'] = aggregate_jobs['input'].apply(
    lambda x: "Asia/Kathmandu" if "nepal" in x else "Indian/Antananarivo"
)
aggregate_jobs['shapefile'] = aggregate_jobs['input'].apply(
    lambda x: "Nepal_Healthsheds2024.zip" if "nepal" in x else "healthsheds2022.zip"
)

aggregate_jobs['hshd_unique_id'] = aggregate_jobs['input'].apply(
    lambda x: "fid" if "nepal" in x else "fs_uid"
)

aggregate_jobs['climate_handler_var'] = aggregate_jobs['variables_short'].apply(
    lambda x: "accum" if x == "tp" else "instant"
)

In [ ]:
aggregate_jobs

Now we add this to the data catalog:

In [ ]:
#| exports:
# update catalog
data_catalog['aggregate']['jobs'].add("jobs_df", aggregate_jobs)

Our data catalog now has an `aggregate|jobs` node with a `jobs_df` entry that contains the dataframe of all the jobs to be run in this task.

In [ ]:
data_catalog['aggregate']['jobs']['jobs_df'].load().head()